# Lab 10

In [1]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

# Seed fix pentru reproductibilitate
SEED = 42
np.random.seed(SEED)

print("Importuri OK. Seed setat la", SEED)

Importuri OK. Seed setat la 42


## 1 - Implementarea mediului `CliffWalkingEnv`

### Grila:
```
[ ][ ][ ][ ][ ][ ][ ][ ][ ][ ][ ][ ]   ← rand 0
[ ][ ][ ][ ][ ][ ][ ][ ][ ][ ][ ][ ]   ← rand 1
[ ][ ][ ][ ][ ][ ][ ][ ][ ][ ][ ][ ]   ← rand 2
[S][C][C][C][C][C][C][C][C][C][C][G]   ← rand 3
 0  1  2  3  4  5  6  7  8  9 10 11
```
- S = Start (3,0), G = Goal (3,11), C = Cliff (3,1..10)

### Codificarea starilor:
Fiecare celula (rand, coloana) devine un numar unic:
```
stare = rand * 12 + coloana
Start = 3*12 + 0  = 36
Goal  = 3*12 + 11 = 47
```

### `done=False` cand cazi in prapastie
Conform specificatiei, episodul continua dupa cliff — agentul e teleportat inapoi la Start, dar nu s-a terminat jocul. Episodul se termina doar la Goal.

In [2]:
class CliffWalkingEnv:
    """
    Mediu determinist: grila 4x12.
    Start: (3,0)  Goal: (3,11)  Cliff: (3,1)..(3,10)
    Acțiuni: 0=UP, 1=DOWN, 2=LEFT, 3=RIGHT
    """

    ROWS = 4
    COLS = 12
    N_STATES  = ROWS * COLS   # 48 stari
    N_ACTIONS = 4             # UP, DOWN, LEFT, RIGHT

    ACTIONS = {
        0: (-1,  0),  # UP
        1: ( 1,  0),  # DOWN
        2: ( 0, -1),  # LEFT
        3: ( 0,  1),  # RIGHT
    }
    ACTION_NAMES = {0: 'UP', 1: 'DOWN', 2: 'LEFT', 3: 'RIGHT'}

    def __init__(self):
        # Starea curenta (numar intre 0 si 47)
        self.state = None
        self.reset()

    # ------------------------------------------------------------------ #
    # Metode helper: conversie intre (rand, col) si numarul starii
    # ------------------------------------------------------------------ #

    def _to_state(self, row, col):
        """Transforma coordonate (rand, col) in numar stare."""
        return row * self.COLS + col

    def _to_coords(self, state):
        """Transforma numar stare in coordonate (rand, col)."""
        return state // self.COLS, state % self.COLS

    def _is_cliff(self, row, col):
        """Verifica daca celula este parte din prapastie."""
        return row == 3 and 1 <= col <= 10

    def _is_goal(self, row, col):
        """Verifica daca celula este Goal."""
        return row == 3 and col == 11

    # ------------------------------------------------------------------ #
    # Interfata standard RL
    # ------------------------------------------------------------------ #

    def reset(self):
        """
        Reseteaza mediul la Start.
        Returneaza: starea initiala (numar intreg)
        """
        self.state = self._to_state(3, 0)  # Start = (3,0)
        return self.state

    def step(self, action):
        """
        Executa o actiune.

        Args:
            action: 0=UP, 1=DOWN, 2=LEFT, 3=RIGHT

        Returns:
            (next_state, reward, done)
            - next_state: starea noua (int)
            - reward: recompensa primita (float)
            - done: True daca am ajuns la Goal
        """
        row, col = self._to_coords(self.state)

        # Calculam noua pozitie
        dr, dc = self.ACTIONS[action]
        new_row = row + dr
        new_col = col + dc

        # Daca iesim din grila, ramanem pe loc (bounce)
        new_row = max(0, min(self.ROWS - 1, new_row))
        new_col = max(0, min(self.COLS - 1, new_col))

        # Verificam ce s-a intamplat
        if self._is_cliff(new_row, new_col):
            # Am cazut in prapastie → penalizare mare, teleportare la Start
            # Episodul CONTINUA (done=False), conform specificatiei
            self.state = self._to_state(3, 0)  # inapoi la Start
            return self.state, -100.0, False

        elif self._is_goal(new_row, new_col):
            # Am ajuns la Goal → episod terminat
            self.state = self._to_state(new_row, new_col)
            return self.state, 0.0, True

        else:
            # Pas normal → penalizare -1
            self.state = self._to_state(new_row, new_col)
            return self.state, -1.0, False

    def render(self):
        """
        Afiseaza grila.
        A = Agent, S = Start, G = Goal, C = Cliff, . = liber
        """
        agent_row, agent_col = self._to_coords(self.state)
        print("+" + "----+" * self.COLS)

        for r in range(self.ROWS):
            row_str = "|"
            for c in range(self.COLS):
                if r == agent_row and c == agent_col:
                    cell = " A  "  # Agentul
                elif r == 3 and c == 0:
                    cell = " S  "  # Start
                elif r == 3 and c == 11:
                    cell = " G  "  # Goal
                elif self._is_cliff(r, c):
                    cell = " C  "  # Cliff
                else:
                    cell = " .  "  # Liber
                row_str += cell + "|"
            print(row_str)

        print("+" + "----+" * self.COLS)
        print(f"Stare curenta: {self.state} = {self._to_coords(self.state)}")
        print()


# ---- TEST ----
env = CliffWalkingEnv()
print("=== Test mediu ===")
print(f"Stare initiala: {env.reset()} (36 = Start)")
env.render()

print("Facem RIGHT (3):")
s, r, done = env.step(3)  # RIGHT → intrăm în Cliff!
print(f"  → stare: {s}, recompensa: {r}, terminat: {done}")
print("  → ar trebui sa fim inapoi la Start (36), reward=-100, done=False")
env.render()

print("Facem UP (0):")
s, r, done = env.step(0)  # UP → mergem la (2,0)
print(f"  → stare: {s}, recompensa: {r}, terminat: {done}")
print("  → ar trebui sa fim la starea 24 = (2,0), reward=-1, done=False")

=== Test mediu ===
Stare initiala: 36 (36 = Start)
+----+----+----+----+----+----+----+----+----+----+----+----+
| .  | .  | .  | .  | .  | .  | .  | .  | .  | .  | .  | .  |
| .  | .  | .  | .  | .  | .  | .  | .  | .  | .  | .  | .  |
| .  | .  | .  | .  | .  | .  | .  | .  | .  | .  | .  | .  |
| A  | C  | C  | C  | C  | C  | C  | C  | C  | C  | C  | G  |
+----+----+----+----+----+----+----+----+----+----+----+----+
Stare curenta: 36 = (3, 0)

Facem RIGHT (3):
  → stare: 36, recompensa: -100.0, terminat: False
  → ar trebui sa fim inapoi la Start (36), reward=-100, done=False
+----+----+----+----+----+----+----+----+----+----+----+----+
| .  | .  | .  | .  | .  | .  | .  | .  | .  | .  | .  | .  |
| .  | .  | .  | .  | .  | .  | .  | .  | .  | .  | .  | .  |
| .  | .  | .  | .  | .  | .  | .  | .  | .  | .  | .  | .  |
| A  | C  | C  | C  | C  | C  | C  | C  | C  | C  | C  | G  |
+----+----+----+----+----+----+----+----+----+----+----+----+
Stare curenta: 36 = (3, 0)

Facem UP (0):


## 2 - Algoritmul Q-learning

### Cum functioneaza?

Algoritmul mentine un **tabel Q** de dimensiune 48×4 (48 stari, 4 actiuni).
Fiecare valoare `Q[s][a]` reprezintă: *"Cat de bun e sa fac actiunea `a` din starea `s`?"*

Initializam cu 0 — agentul nu stie nimic la inceput.

### (explorare vs. exploatare)
- Cu probabilitate **epsilon = 0.1** → alegem o actiune **aleatoare** (exploram)
- Cu probabilitate **1-epsilon = 0.9** → alegem actiunea cu cel mai mare Q (exploatam)

### Regula de update Q-learning:
```
Q(s,a)  =   Q(s,a)  +  α · [ td_target  -   Q(s,a) ]
  ↑            ↑       ↑        ↑              ↑
valoarea    valoarea  pas    ce AR trebui    ce CREDEAM
 nouă        veche    mic    sa fie          ca e

α = cat de mult corectam dintr-o data (0.1 = 10% din eroare)
td_target = r + γ · max Q(s')
                          ↑
              Folosim MAXIMUL din starea urmatoare
r = recompensa primita pentru actiunea facuta
γ = cat de mult conteaza viitorul

```
Aceasta e diferenta fata de SARSA.

In [3]:
def q_learning(env, n_episodes=500, alpha=0.1, gamma=1.0, epsilon=0.1, seed=SEED):
    """
    Implementare Q-learning

    Args:
        env      : mediul CliffWalkingEnv
        n_episodes: numar de episoade de antrenament
        alpha    : rata de invatare (cat de repede uitam trecutul)
        gamma    : factorul de discount (1.0 = recompensele viitoare conteaza la fel)
        epsilon  : probabilitatea de explorare aleatoare
        seed     : seed pentru reproductibilitate

    Returns:
        Q          : tabelul Q final (48x4)
        rewards_per_episode : lista recompenselor totale per episod
    """
    rng = np.random.default_rng(seed)  # generator aleatoriu propriu, nu afectează global

    # Initializam tabelul Q cu zerouri
    # Dimensiune: N_STATES x N_ACTIONS = 48 x 4
    Q = np.zeros((env.N_STATES, env.N_ACTIONS))

    rewards_per_episode = []  # pentru grafic

    for episode in range(n_episodes):
        state = env.reset()       # pornim din Start
        total_reward = 0          # suma recompenselor din episodul acesta
        done = False

        while not done:
            if rng.random() < epsilon:
                # EXPLORARE: actiune aleatoare
                action = rng.integers(0, env.N_ACTIONS)
            else:
                # EXPLOATARE: actiunea cu Q maxim
                # argmax returneaza indexul celei mai mari valori
                action = int(np.argmax(Q[state]))

            # Executam actiunea in mediu
            next_state, reward, done = env.step(action)
            total_reward += reward

            # Actualizam Q (regula Q-learning)
            # Tinta: r + γ * max_a'(Q[next_state])
            # Daca episodul s-a terminat (goal), nu mai există stare viitoare
            if done:
                td_target = reward                         # nu mai adaugam valori viitoare
            else:
                td_target = reward + gamma * np.max(Q[next_state])  # luam maximul

            # Eroarea (cat de gresit era Q-ul)
            td_error = td_target - Q[state, action]

            # Actualizam: facem un pas mic (alpha) in directia erorii
            Q[state, action] += alpha * td_error

            # Trecem la starea următoare
            state = next_state

        rewards_per_episode.append(total_reward)

    return Q, rewards_per_episode

## 2 — Algoritmul SARSA

### Ce e diferit fata de Q-learning?

SARSA nu ia maximul, alege actiunea urmatoare si invata ca daca merge aproape de cliff, exista sansa reala sa explorez random si sa mor.

### Regula de update SARSA:
```
Q(s,a) ← Q(s,a) + α · [r + γ · Q(s', a') - Q(s,a)]
                              ↑
              Folosim actiunea a' pe care CHIAR O VOM FACE
              (inclusiv explorarea aleatoare)
```

In [4]:
def sarsa(env, n_episodes=500, alpha=0.1, gamma=1.0, epsilon=0.1, seed=SEED):
    """
    Implementare SARSA

    Singura diferență față de Q-learning:
    - Alegem a' INAINTE de update (nu folosim max)
    - Folosim Q[next_state, next_action] in loc de max(Q[next_state])
    """
    rng = np.random.default_rng(seed)

    Q = np.zeros((env.N_STATES, env.N_ACTIONS))
    rewards_per_episode = []

    for episode in range(n_episodes):
        state = env.reset()
        total_reward = 0
        done = False

        # SARSA: alegem prima acțiune INAINTE de bucla while
        # (avem nevoie de a' la sfarsitul fiecarui pas)
        if rng.random() < epsilon:
            action = rng.integers(0, env.N_ACTIONS)
        else:
            action = int(np.argmax(Q[state]))

        while not done:
            # --- Executam actiunea ---
            next_state, reward, done = env.step(action)
            total_reward += reward

            # --- Alegem ACTIUNEA URMATOARE (a') ---
            if done:
                next_action = None  # nu mai contează
            else:
                if rng.random() < epsilon:
                    next_action = rng.integers(0, env.N_ACTIONS)  # explorare
                else:
                    next_action = int(np.argmax(Q[next_state]))    # exploatare

            # --- Actualizam Q (regula SARSA) ---
            if done:
                td_target = reward                                   # fara viitor
            else:
                td_target = reward + gamma * Q[next_state, next_action]  # actiunea efectiva

            td_error = td_target - Q[state, action]
            Q[state, action] += alpha * td_error

            # Trecem la (starea, actiunea) urmatoare
            state = next_state
            action = next_action

        rewards_per_episode.append(total_reward)

    return Q, rewards_per_episode

## 3 — Experimentul central

Rulam ambii algoritmi, generam graficele si vizualizam politicile.

In [5]:
# ---- Rulam ambii algoritmi ----
env = CliffWalkingEnv()

print("Antrenam Q-learning...")
Q_ql, rewards_ql = q_learning(env, n_episodes=500)
print("Q-learning terminat.")

print("Antrenam SARSA...")
Q_sarsa, rewards_sarsa = sarsa(env, n_episodes=500)
print("SARSA terminat.")

print(f"\nRecompensa medie ultimele 50 ep. — Q-learning: {np.mean(rewards_ql[-50:]):.1f}")
print(f"Recompensa medie ultimele 50 ep. — SARSA:      {np.mean(rewards_sarsa[-50:]):.1f}")

Antrenam Q-learning...
Q-learning terminat.
Antrenam SARSA...
SARSA terminat.

Recompensa medie ultimele 50 ep. — Q-learning: -52.7
Recompensa medie ultimele 50 ep. — SARSA:      -18.1


In [6]:
# ---- Grafic: recompense per episod ----

def moving_average(data, window=10):
    return np.convolve(data, np.ones(window) / window, mode='valid')

ma_ql    = moving_average(rewards_ql,    window=10)
ma_sarsa = moving_average(rewards_sarsa, window=10)

fig, ax = plt.subplots(figsize=(12, 5))

# Linii transparente pentru datele brute
ax.plot(rewards_ql,    alpha=0.2, color='steelblue')
ax.plot(rewards_sarsa, alpha=0.2, color='darkorange')

# Medii mobile — mai clare
ax.plot(ma_ql,    label='Q-learning (medie mobilă 10)', color='steelblue',  linewidth=2)
ax.plot(ma_sarsa, label='SARSA (medie mobilă 10)',       color='darkorange', linewidth=2)

ax.axhline(y=-13, color='green', linestyle='--', alpha=0.5, label='Drum optim teoretic (-13)')

ax.set_xlabel('Episod')
ax.set_ylabel('Recompensa totala')
ax.set_title('Q-learning vs. SARSA — Recompensa per episod (CliffWalking)')
ax.legend()
ax.grid(alpha=0.3)
ax.set_ylim(-200, 10)

plt.tight_layout()
plt.savefig('grafic_recompense.png', dpi=150)
plt.show()
print("Grafic salvat ca 'grafic_recompense.png'")

Grafic salvat ca 'grafic_recompense.png'


In [7]:
# Vizualizarea politicilor greedy

def get_greedy_policy(Q, n_rows=4, n_cols=12):
    """
    Extrage politica greedy din tabelul Q:
    pentru fiecare stare, ia actiunea cu Q maxim.
    """
    policy = np.argmax(Q, axis=1)  # forma: (48,)
    return policy.reshape(n_rows, n_cols)


def plot_policy(Q, title, ax):
    """
    Afiseaza politica greedy pe grila cu sageti.
    """
    ARROW = {0: '↑', 1: '↓', 2: '←', 3: '→'}  # UP, DOWN, LEFT, RIGHT
    ROWS, COLS = 4, 12
    policy = get_greedy_policy(Q)

    # Colorăm fundalul grilei
    grid_colors = np.zeros((ROWS, COLS, 3))
    for r in range(ROWS):
        for c in range(COLS):
            if r == 3 and c == 0:         # Start
                grid_colors[r, c] = [0.2, 0.8, 0.2]   # verde
            elif r == 3 and c == 11:      # Goal
                grid_colors[r, c] = [0.2, 0.2, 0.8]   # albastru
            elif r == 3 and 1 <= c <= 10: # Cliff
                grid_colors[r, c] = [0.8, 0.2, 0.2]   # roșu
            else:
                grid_colors[r, c] = [0.95, 0.95, 0.95] # gri deschis

    ax.imshow(grid_colors, aspect='equal', interpolation='none')

    # Săgeți și etichete
    for r in range(ROWS):
        for c in range(COLS):
            if r == 3 and c == 0:
                ax.text(c, r, 'S', ha='center', va='center', fontsize=12, fontweight='bold', color='white')
            elif r == 3 and c == 11:
                ax.text(c, r, 'G', ha='center', va='center', fontsize=12, fontweight='bold', color='white')
            elif r == 3 and 1 <= c <= 10:
                ax.text(c, r, 'C', ha='center', va='center', fontsize=10, color='white')
            else:
                ax.text(c, r, ARROW[policy[r, c]], ha='center', va='center',
                        fontsize=16, color='black')

    ax.set_xticks(range(COLS))
    ax.set_yticks(range(ROWS))
    ax.set_xticklabels(range(COLS))
    ax.set_yticklabels(range(ROWS))
    ax.set_title(title, fontsize=13, fontweight='bold')
    ax.grid(color='gray', linewidth=0.5)


fig, axes = plt.subplots(1, 2, figsize=(16, 4))
plot_policy(Q_ql,    'Politica greedy — Q-learning', axes[0])
plot_policy(Q_sarsa, 'Politica greedy — SARSA',       axes[1])

plt.suptitle('Comparație politici: Q-learning vs SARSA', fontsize=14, y=1.02)
plt.tight_layout()
plt.savefig('politici_comparatie.png', dpi=150, bbox_inches='tight')
plt.show()
print("Grafic salvat ca 'politici_comparatie.png'")

# Afișăm și drumul optim găsit de fiecare
def trace_path(env, Q, max_steps=50):
    """Urmărește politica greedy și returnează drumul."""
    state = env.reset()
    path = [env._to_coords(state)]
    for _ in range(max_steps):
        action = int(np.argmax(Q[state]))
        state, _, done = env.step(action)
        path.append(env._to_coords(state))
        if done:
            break
    return path

path_ql    = trace_path(env, Q_ql)
path_sarsa = trace_path(env, Q_sarsa)
print(f"\nDrum Q-learning ({len(path_ql)-1} pași): {path_ql}")
print(f"Drum SARSA      ({len(path_sarsa)-1} pași): {path_sarsa}")

Grafic salvat ca 'politici_comparatie.png'

Drum Q-learning (13 pași): [(3, 0), (2, 0), (2, 1), (2, 2), (2, 3), (2, 4), (2, 5), (2, 6), (2, 7), (2, 8), (2, 9), (2, 10), (2, 11), (3, 11)]
Drum SARSA      (15 pași): [(3, 0), (2, 0), (1, 0), (1, 1), (1, 2), (1, 3), (1, 4), (1, 5), (1, 6), (1, 7), (1, 8), (1, 9), (1, 10), (1, 11), (2, 11), (3, 11)]


---
## Raspunsuri

### (a) Care algoritm invata un drum mai scurt? Care obtine recompensa mai mare in timpul antrenarii?

**Q-learning** invata un drum mai scurt in politica greedy finala. Drumul optim pe langa prapastie are **13 pasi** (sus, 11× dreapta, jos), acumuland recompensa -13. Q-learning, fiind *off-policy*, invata Q-valorile optime indiferent de explorare — la finalul antrenarii, politica sa greedy merge direct pe marginea prapastiei.

**SARSA**, insa, obtine o **recompensa cumulata mai mare in timpul antrenarii**. Desi drumul sau final e mai lung (merge pe randul 1 sau 2, ocolind prapastia), evita caderile accidentale de -100 cauzate de explorare. Graficul arata ca suma recompenselor per episod a lui SARSA e mai putin negativa decat cea a Q-learning pe parcursul antrenamentului.

### (b) De ce converg la politici diferite?

Diferenta fundamentala este in **tinta folosita pentru update**:

- **Q-learning (off-policy)**: tinta este r + γ · max_{a'} Q(s', a'). Algoritmul presupune ca din starea urmatoare va fi ales **maximul** — adica va actiona perfect. Nu tine cont ca de fapt, cu probabilitate ε, va explora aleator si ar putea cadea in prapastie. Prin urmare, Q-learning evalueaza politica *optima*, nu cea executata. Aceasta ii permite sa invete valorile drumului scurt pe langa cliff, ignorand riscul explorarii.

- **SARSA (on-policy)**: tinta este r + γ · Q(s', a'), unde `a'` este actiunea **efectiv aleasa** cu ε-greedy. Daca politica alege uneori aleator (ε=0.1), acea posibilitate de cadere este *inclusa in calcul*. Celulele de langa prapastie au o valoare Q mai mica in SARSA, deoarece algoritmul stie ca uneori va cadea. Rezultatul: agentul prefera drumul mai sigur, mai departe de cliff.

Acesta este exact sensul distinctiei **on-policy vs. off-policy**: SARSA evalueaza si imbunatateste politica *pe care o executa*, Q-learning evalueaza politica *optima* indiferent de ce executa.

### (c) De ce nu exista o contradictie intre (a) si (b)?

Nu exista contradictie pentru ca cele doua afirmatii masoara lucruri diferite:

- Q-learning gaseste politica **teoretic optima** (drum scurt, recompensa maxima daca ε=0). Dar *in timpul antrenarii*, cu ε > 0, merge pe marginea prapastiei si cade des → recompensa medie mai slaba per episod.

- SARSA gaseste politica **optima pentru agentul care exploreaza** (ε-greedy optimal). Drumul e mai lung, dar agentul cade mai rar → recompensa medie mai buna per episod de antrenament.

Cu alte cuvinte: Q-learning e mai bun *daca vom dezactiva explorarea*, SARSA e mai bun *in timp ce exploreaza*. Ambele afirmatii sunt adevarate simultan.